# Module 06: MLflow Observability

Agents are non-deterministic: the same prompt can yield 3 steps or 8 steps, good answers or hallucinations.

Without logging, you can't answer: "Why did run B perform better than run A?"

MLflow gives you a structured experiment record:
- **params** — what you configured (model, max_steps, agent_type)
- **metrics** — what happened numerically (duration, steps taken)
- **artifacts** — the actual outputs (result text, step traces)

This is how you move from "I hope it works" to "I know it works, here's the data."

## Setup

> **Before running this notebook:** Open a separate terminal and run:
> ```bash
> cd smolagents
> uv run mlflow ui --port 5000
> ```
> Then open http://localhost:5000 in your browser.

In [ ]:
# Install required packages
# Uncomment the line below if running in Google Colab or a fresh environment
# !uv pip install smolagents python-dotenv duckduckgo-search mlflow
# Or using pip:
# !pip install smolagents python-dotenv duckduckgo-search mlflow

In [ ]:
import os

# ----- HF_TOKEN Setup -----
# Option A: Load from .env file (local development)
# from dotenv import load_dotenv
# load_dotenv()

# Option B: Google Colab Secrets
# Uncomment the lines below when running in Google Colab.
# Go to: Colab → Secrets (🔑 icon) → Add HF_TOKEN
# from google.colab import userdata
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# Option C: Set directly (not recommended for shared notebooks)
# os.environ['HF_TOKEN'] = 'hf_your_token_here'

In [ ]:
import os
import time
import json
import mlflow
from dotenv import load_dotenv
from smolagents import CodeAgent, ToolCallingAgent, InferenceClientModel, DuckDuckGoSearchTool

load_dotenv()

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=os.environ["HF_TOKEN"],
)

# Point to local MLflow server
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("smolagents-course")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Ready.")

## Manual Run Logging

The simplest approach: wrap your agent run in `mlflow.start_run()` and log what matters.

In [ ]:
task = "What are the top 3 open-source LLM frameworks in 2024? List them with a one-line description each."

with mlflow.start_run(run_name="baseline-codeagent"):
    # Log configuration (what we set up)
    mlflow.log_param("model_id", "Qwen/Qwen2.5-Coder-32B-Instruct")
    mlflow.log_param("agent_type", "CodeAgent")
    mlflow.log_param("max_steps", 6)
    mlflow.log_param("has_search", True)
    mlflow.log_param("task", task[:100])  # truncate long tasks

    # Run the agent
    agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=model, max_steps=6)
    start = time.time()
    result = agent.run(task)
    elapsed = time.time() - start

    # Log what happened
    mlflow.log_metric("duration_seconds", round(elapsed, 2))
    mlflow.log_metric("steps_taken", len(agent.memory.steps))
    mlflow.log_text(str(result), "result.txt")

    print(f"Logged run: {elapsed:.1f}s, {len(agent.memory.steps)} steps")
    print("Result:", result[:300])

## Step-Level Tracing

Logging just the final result isn't enough for debugging. We want to capture every step the agent took — what it thought, what it called, what it got back.

In [ ]:
def run_and_trace(agent, task: str, run_name: str, **params) -> str:
    """
    Run an agent on a task and log everything to MLflow.
    
    Logs: all params, duration_seconds, steps_taken metrics,
    full result as artifact, and step-by-step trace as JSON artifact.
    """
    with mlflow.start_run(run_name=run_name):
        # Log all provided params
        mlflow.log_params({"task_preview": task[:100], **params})

        # Run
        start = time.time()
        result = agent.run(task)
        elapsed = time.time() - start

        # Capture step trace
        steps_data = []
        for i, step in enumerate(agent.memory.steps):
            steps_data.append({
                "index": i,
                "type": type(step).__name__,
                "content": str(step)[:500],
            })

        # Log metrics and artifacts
        mlflow.log_metric("duration_seconds", round(elapsed, 2))
        mlflow.log_metric("steps_taken", len(agent.memory.steps))
        mlflow.log_text(str(result), "result.txt")
        mlflow.log_dict({"steps": steps_data}, "steps_trace.json")

        return result

# Test it
result = run_and_trace(
    agent=CodeAgent(tools=[DuckDuckGoSearchTool()], model=model, max_steps=6),
    task=task,
    run_name="traced-codeagent",
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    agent_type="CodeAgent",
    max_steps=6,
)
print("Traced run complete. Check http://localhost:5000")

## Comparing Agent Types

Now let's run the same task with `CodeAgent` and `ToolCallingAgent` and compare in MLflow.

In [ ]:
# Run 1: CodeAgent
run_and_trace(
    agent=CodeAgent(tools=[DuckDuckGoSearchTool()], model=model, max_steps=6),
    task=task,
    run_name="compare-CodeAgent",
    agent_type="CodeAgent",
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    max_steps=6,
)

# Brief pause to avoid rate limits
time.sleep(3)

# Run 2: ToolCallingAgent — same task, same model
run_and_trace(
    agent=ToolCallingAgent(tools=[DuckDuckGoSearchTool()], model=model, max_steps=6),
    task=task,
    run_name="compare-ToolCallingAgent",
    agent_type="ToolCallingAgent",
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct",
    max_steps=6,
)

print("Both runs logged.")
print("Go to http://localhost:5000 → smolagents-course experiment")
print("Select both runs → click 'Compare' to see side-by-side metrics")

## Navigating the MLflow UI

Here's what to do in your browser at http://localhost:5000:

1. **Open the experiment**: Click `smolagents-course` in the left sidebar
2. **See all runs**: Each row is one agent run, with params and metrics columns visible at a glance
3. **Compare two runs**: Check the boxes next to two runs → click the **Compare** button
4. **View artifacts**: Click a run name → scroll to **Artifacts** section → click `result.txt` or `steps_trace.json`
5. **Filter runs**: Use the search bar to filter by param values (e.g., `params.agent_type = 'CodeAgent'`)

> **Exercise preview**: After running the comparison cells above, compare the `duration_seconds` and `steps_taken` for CodeAgent vs ToolCallingAgent. Which was faster? Which used fewer steps?

## Exercises

In [ ]:
# TODO Exercise 1: Log the multi-agent system from Module 05
# 
# Re-create the manager + specialist setup from Module 05 (copy the code).
# Wrap the manager.run() call using run_and_trace().
# 
# Additional challenge: also log the SPECIALIST step counts as separate metrics:
#   mlflow.log_metric("web_researcher_steps", len(web_researcher.memory.steps))
#   mlflow.log_metric("data_analyst_steps", len(data_analyst.memory.steps))
# 
# Note: you'll need to do this INSIDE a mlflow.start_run() block manually,
# since run_and_trace() doesn't know about specialist agents.
# 
# After running, inspect the artifacts — can you see which specialist did what?

# Your code here:

In [ ]:
# TODO Exercise 2: Non-determinism experiment
# 
# Run the SAME agent and task 3 times. Log each as a separate MLflow run.
# Use run names like: "nondeterminism-run-1", "nondeterminism-run-2", "nondeterminism-run-3"
# 
# After all 3 runs:
# 1. Open MLflow UI and compare all 3 runs
# 2. Compare: steps_taken and duration_seconds across the 3 runs
# 3. Download result.txt from each run
# 4. Are the answers identical? Different wording but same facts? Completely different?
# 
# This is the empirical demonstration of agent non-determinism.
# Note your observations as comments below.

# Your code here:

# Observations:
# Run 1 steps: ?
# Run 2 steps: ?
# Run 3 steps: ?
# Were results consistent? (yes/no/partially):

## What You Built

You now know how to:
- Start a local MLflow tracking server and connect to it from a notebook
- Log agent runs with `params` (configuration), `metrics` (measurements), and `artifacts` (outputs)
- Capture step-level traces as JSON artifacts for deep debugging
- Compare runs in the MLflow UI to measure agent performance differences
- Apply observability to multi-agent systems by tracking specialist metrics

**Key insight:** Logging transforms agent development from guesswork to engineering. When a run fails or degrades, your first question is now "what do the logs say?" not "why did this happen?"

## Course Wrap-Up

### What you built across 6 modules:

| Module | What You Learned |
|--------|------------------|
| 01 Foundations | The agent loop, CodeAgent, step inspection |
| 02 Tools | @tool decorator, Tool subclass, schema design |
| 03 Agent Types | CodeAgent vs ToolCallingAgent, when to use each, model agnosticism |
| 04 Web Search | DuckDuckGo, VisitWebpage, research workflows |
| 05 Multi-Agent | Manager + specialists, delegation, design principles |
| 06 Observability | MLflow logging, tracing, run comparison |

### Where to go next:

- **smolagents docs**: https://huggingface.co/docs/smolagents
- **HuggingFace Agents Course**: https://huggingface.co/learn/agents-course
- **MLflow LLM Tracing**: https://mlflow.org/docs/latest/llms/tracing/index.html

### Ideas for extending what you built:
- Deploy an agent as a Gradio app
- Add memory/persistence between agent runs
- Use a local model via Ollama (no API key needed)
- Build a Slack bot powered by a multi-agent system
- Add automated evaluation: compare agent outputs against expected answers